# Depth Estimation Visualization
## Side-by-Side Comparison of Depth Models

This notebook visualizes depth estimation results from multiple models.

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

import numpy as np
import cv2
import matplotlib.pyplot as plt
import json
from common import DepthVisualizer, PoseVisualizer

# Configure matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 10)

print('✓ Visualization setup complete')

## Load Results

In [ ]:
# Load benchmark results
results_path = Path('../outputs/metrics/benchmark_results_results.json')

if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print(f'✓ Loaded results from {results_path}')
    print(f'Models: {list(results.keys())}')
else:
    print(f'Results file not found: {results_path}')
    results = {}

## Create Comparison Tables

In [ ]:
import pandas as pd

# Create comparison table
rows = []
for model_name, metrics in results.items():
    if model_name == 'pose':
        continue
    
    row = {'Model': model_name}
    
    if 'depth_metrics' in metrics:
        for metric_name, metric_val in metrics['depth_metrics'].items():
            if isinstance(metric_val, dict):
                row[f'{metric_name}_mean'] = metric_val.get('mean', np.nan)
            else:
                row[metric_name] = metric_val
    
    if 'temporal_metrics' in metrics:
        row.update(metrics['temporal_metrics'])
    
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string())

## Plot Metrics

In [ ]:
# Plot key metrics
if not df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    metrics_to_plot = ['mae_mean', 'rmse_mean', 'abs_rel_mean', 'silog_mean']
    available_metrics = [m for m in metrics_to_plot if m in df.columns]
    
    for idx, (ax, metric) in enumerate(zip(axes.flat, available_metrics[:4])):
        df_sorted = df.sort_values(metric)
        ax.barh(df_sorted['Model'], df_sorted[metric])
        ax.set_xlabel(metric.replace('_mean', ''))
        ax.set_title(f'Model Comparison: {metric}')
        ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../outputs/visualizations/metrics_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Metrics comparison plot saved')

## Export Summary Statistics

In [ ]:
# Summary statistics
summary = {
    'num_models': len([k for k in results.keys() if k != 'pose']),
    'models': [k for k in results.keys() if k != 'pose'],
    'evaluation_date': str(pd.Timestamp.now()),
}

# Find best models
for metric in ['mae_mean', 'rmse_mean', 'abs_rel_mean', 'silog_mean']:
    if metric in df.columns:
        best_idx = df[metric].idxmin()
        best_model = df.loc[best_idx, 'Model']
        best_value = df.loc[best_idx, metric]
        summary[f'best_{metric}'] = {'model': best_model, 'value': best_value}

# Save summary
summary_path = Path('../outputs/metrics/summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print('✓ Summary saved')
print(json.dumps(summary, indent=2, default=str))